# Import Libraries

In [59]:
import os
from pathlib import Path

from tqdm import tqdm

import numpy as np
import pandas as pd

import librosa

import matplotlib.pyplot as plt
import seaborn as sns

# Import initial dataset

In [60]:
init_df = pd.read_csv("../data/metadata/metadata_compiled_init.csv")
init_df.head()

,uuid,datetime,cough_detected,SNR,latitude,longitude,age,gender,respiratory_condition,fever_muscle_pain,...,quality_4,cough_type_4,dyspnea_4,wheezing_4,stridor_4,choking_4,congestion_4,nothing_4,diagnosis_4,severity_4
0,00014dcc-0f06-4c27-8c7b-737b18a2cf4c,2020-11-25T18:58:50.488301+00:00,0.0155,7.326171,48.9,2.4,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00039425-7f3a-42aa-ac13-834aaa2b6b92,2020-04-13T21:30:59.801831+00:00,0.9609,16.151433,31.3,34.8,15.0,male,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0007c6f1-5441-40e6-9aaf-a761d8f2da3b,2020-10-18T15:38:38.205870+00:00,0.1643,16.217201,NaN,NaN,46.0,female,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0009eb28-d8be-4dc1-92bb-907e53bc5c7a,2020-04-12T04:02:18.159383+00:00,0.9301,20.146058,40.0,-75.1,34.0,male,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0012c608-33d0-4ef7-bde3-75a0b1a0024e,2020-04-15T01:03:59.029326+00:00,0.0482,0.000000,-16.5,-71.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
init_df.columns

Index(['uuid', 'datetime', 'cough_detected', 'SNR', 'latitude', 'longitude',
       'age', 'gender', 'respiratory_condition', 'fever_muscle_pain', 'status',
       'quality_1', 'cough_type_1', 'dyspnea_1', 'wheezing_1', 'stridor_1',
       'choking_1', 'congestion_1', 'nothing_1', 'diagnosis_1', 'severity_1',
       'quality_2', 'cough_type_2', 'dyspnea_2', 'wheezing_2', 'stridor_2',
       'choking_2', 'congestion_2', 'nothing_2', 'diagnosis_2', 'severity_2',
       'quality_3', 'cough_type_3', 'dyspnea_3', 'wheezing_3', 'stridor_3',
       'choking_3', 'congestion_3', 'nothing_3', 'diagnosis_3', 'severity_3',
       'quality_4', 'cough_type_4', 'dyspnea_4', 'wheezing_4', 'stridor_4',
       'choking_4', 'congestion_4', 'nothing_4', 'diagnosis_4', 'severity_4'],
      dtype='str')

In [62]:
init_df.shape

(27550, 51)

# Drop rows with less than 0.5 cough_detected coefficient

In [63]:
small_cough_detected = (init_df['cough_detected'] < 0.5).sum()
print(small_cough_detected)

9324


In [64]:
mask = (init_df['cough_detected'] < 0.5) & (init_df['status'].isin(['COVID-19', 'symptomatic']))
print(init_df[mask].shape[0])

881


In [65]:
mask = (init_df['cough_detected'] < 0.3) & (init_df['status'].isin(['COVID-19', 'symptomatic']))
print(init_df[mask].shape[0])

720


# Assuming that there could be some errors, decided to drop rows with less than 0.4 cough_detected cooefficient

In [66]:
init_df.drop(init_df[init_df['cough_detected'] < 0.4].index, inplace=True)

In [67]:
init_df.shape

(19001, 51)

In [68]:
nan_cough_detected = (init_df['cough_detected'].isnull()).sum()
print(nan_cough_detected)

0


# Check what is the percentage of null values in each column

In [69]:
null_col = init_df.isnull().mean() * 100
print(null_col.sort_values(ascending=False).to_string())

diagnosis_4              95.847587
diagnosis_3              95.831798
cough_type_3             95.816010
severity_3               95.816010
wheezing_3               95.810747
dyspnea_3                95.810747
stridor_3                95.810747
quality_3                95.810747
nothing_3                95.810747
choking_3                95.810747
congestion_3             95.810747
severity_4               95.794958
cough_type_2             95.784432
cough_type_4             95.784432
dyspnea_1                95.779170
wheezing_1               95.779170
choking_1                95.779170
congestion_1             95.779170
nothing_1                95.779170
congestion_2             95.779170
nothing_2                95.779170
diagnosis_2              95.779170
severity_2               95.779170
dyspnea_2                95.779170
wheezing_2               95.779170
stridor_2                95.779170
choking_2                95.779170
quality_2                95.779170
cough_type_1        

# Drop anything higher than 40%

In [70]:
init_df = init_df.loc[:, init_df.isnull().mean() * 100 < 90]

In [71]:
init_df.columns

Index(['uuid', 'datetime', 'cough_detected', 'SNR', 'latitude', 'longitude',
       'age', 'gender', 'respiratory_condition', 'fever_muscle_pain',
       'status'],
      dtype='str')

In [72]:
init_df = init_df.dropna(subset=['status'])

In [73]:
init_df.shape

(13053, 11)

# Drop datetime column

In [74]:
init_df.drop(columns=['datetime'], inplace=True)

In [75]:
init_df.columns

Index(['uuid', 'cough_detected', 'SNR', 'latitude', 'longitude', 'age',
       'gender', 'respiratory_condition', 'fever_muscle_pain', 'status'],
      dtype='str')

# Check what is the percentage of null values in the columns

In [76]:
null_col = init_df.isnull().mean() * 100
print(null_col.sort_values(ascending=False).to_string())

longitude                39.469854
latitude                 39.469854
age                       5.048648
uuid                      0.000000
cough_detected            0.000000
SNR                       0.000000
gender                    0.000000
respiratory_condition     0.000000
fever_muscle_pain         0.000000
status                    0.000000


# Check proportion of labels

In [77]:
init_df['status'].value_counts(normalize=True) * 100

status
healthy        77.514748
symptomatic    16.356393
COVID-19        6.128859
Name: proportion, dtype: float64